# Confounding Examples - Solutions

Complete solutions to the exercises in `confounding_exercises.ipynb`.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from typing import List, Tuple
from sklearn.linear_model import LinearRegression, LogisticRegression

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

In [ ]:
# Helper function
def draw_dag(edges: List[Tuple[str, str]], title: str = "DAG", 
             node_size: int = 3000, font_size: int = 11, 
             figsize: Tuple[int, int] = (10, 6)):
    G = nx.DiGraph()
    G.add_edges_from(edges)
    
    plt.figure(figsize=figsize)
    pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    
    nx.draw(G, pos, 
            with_labels=True,
            node_color='lightblue',
            node_size=node_size,
            font_size=font_size,
            font_weight='bold',
            arrows=True,
            arrowsize=20,
            arrowstyle='->',
            edge_color='gray',
            width=2,
            connectionstyle='arc3,rad=0.1')
    
    plt.title(title, fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    return G

---

## Exercise 1 Solution: Identify Confounders

**Research Question:** Does remote work increase employee productivity?

In [ ]:
# Draw the DAG
edges_remote = [
    # Confounders
    ('Job_Autonomy', 'Remote_Work'),
    ('Job_Autonomy', 'Productivity'),
    ('Experience', 'Remote_Work'),
    ('Experience', 'Productivity'),
    ('Self_Discipline', 'Remote_Work'),
    ('Self_Discipline', 'Productivity'),
    
    # Direct effect (what we want to estimate)
    ('Remote_Work', 'Productivity')
]

G_remote = draw_dag(edges_remote, 
                    title="Remote Work → Productivity with Confounders",
                    figsize=(11, 7))

### Confounders Explanation

**Confounder 1: Job Autonomy**
- **Why it affects treatment (Remote Work):** 
  - Employees with autonomous jobs (e.g., software developers, consultants) have more flexibility and are more likely to be allowed to work remotely
  - Jobs requiring constant supervision are less likely to offer remote options
  
- **Why it affects outcome (Productivity):** 
  - Autonomous jobs naturally lead to higher productivity because employees can organize their work optimally
  - Self-directed work is associated with higher motivation and output
  
- **Why it's NOT a mediator:** 
  - Job autonomy is a pre-existing characteristic of the role
  - Remote work doesn't create autonomy; autonomy determines who gets remote work
  - The causal order is: Autonomy → Remote Work, not Remote Work → Autonomy

**Confounder 2: Experience (Seniority)**
- **Why it affects treatment (Remote Work):** 
  - Senior employees with more experience are trusted to work remotely
  - Junior employees often required to be in office for training and supervision
  
- **Why it affects outcome (Productivity):** 
  - Experienced workers are more productive due to accumulated skills and knowledge
  - They work more efficiently regardless of location
  
- **Why it's NOT a mediator:** 
  - Experience is gained over time, not caused by remote work
  - Remote work doesn't make someone experienced; experience determines who gets remote privileges

**Confounder 3: Self-Discipline**
- **Why it affects treatment (Remote Work):** 
  - Managers grant remote work to employees who have demonstrated self-discipline
  - Self-directed individuals actively seek and obtain remote arrangements
  
- **Why it affects outcome (Productivity):** 
  - Disciplined employees are productive regardless of where they work
  - They manage time well, stay focused, and meet deadlines
  
- **Why it's NOT a mediator:** 
  - Self-discipline is a personality trait that exists before remote work
  - Remote work doesn't create discipline (though it may test it)

### Backdoor Paths

1. Remote_Work ← Job_Autonomy → Productivity
2. Remote_Work ← Experience → Productivity  
3. Remote_Work ← Self_Discipline → Productivity

**Minimal adjustment set:** Control for {Job_Autonomy, Experience, Self_Discipline}

All three confounders must be controlled to block all backdoor paths and get an unbiased estimate of the causal effect of remote work on productivity.

---

## Exercise 2 Solution: Simpson's Paradox

**Scenario:** Drug effectiveness across disease severity levels

In [ ]:
def simulate_simpsons_paradox(n: int = 1000) -> pd.DataFrame:
    """
    Simulate drug trial where disease severity confounds the relationship.
    
    Scenario:
    - Sicker patients are more likely to receive the drug (treatment selection)
    - Sicker patients have worse outcomes regardless of treatment
    - Drug actually helps, but appears harmful overall due to confounding
    """
    np.random.seed(42)
    
    # Disease severity (confounder) - ranges 0-10
    severity = np.random.uniform(0, 10, n)
    
    # Drug treatment - sicker patients more likely to get drug
    # Probability increases with severity
    prob_treatment = 0.2 + 0.06 * severity  # Ranges from 20% to 80%
    treatment = np.random.binomial(1, prob_treatment)
    
    # Recovery score (0-100, higher is better)
    # - Drug helps (+15 points)
    # - But severity hurts (-8 points per severity unit)
    recovery = (80  # Baseline
               + 15 * treatment  # Drug HELPS (true positive effect)
               - 8 * severity  # Severity HURTS (strong negative effect)
               + np.random.normal(0, 5, n))
    recovery = np.clip(recovery, 0, 100)
    
    return pd.DataFrame({
        'severity': severity,
        'treatment': treatment,
        'recovery': recovery
    })

df = simulate_simpsons_paradox()

print("Simpson's Paradox: Drug Effectiveness")
print("="*50)
print("\nScenario: Drug trial with disease severity as confounder")
print("  • Sicker patients more likely to receive drug")
print("  • Sicker patients have worse outcomes")
print("  • Drug actually helps, but confounding masks this!")

print(f"\nData summary:")
print(f"  N = {len(df)}")
print(f"  Treatment rate: {df['treatment'].mean():.1%}")
print(f"  Mean severity (treated): {df[df['treatment']==1]['severity'].mean():.2f}")
print(f"  Mean severity (control): {df[df['treatment']==0]['severity'].mean():.2f}")

In [ ]:
# Calculate naive and adjusted estimates
y = df['recovery'].values

# Naive estimate (biased)
X_naive = df[['treatment']].values
model_naive = LinearRegression().fit(X_naive, y)
effect_naive = model_naive.coef_[0]

# Adjusted estimate (controlling for severity)
X_adjusted = df[['treatment', 'severity']].values
model_adjusted = LinearRegression().fit(X_adjusted, y)
effect_adjusted = model_adjusted.coef_[0]

print("\n" + "="*50)
print("SIMPSON'S PARADOX DEMONSTRATION")
print("="*50)

print(f"\nTRUE drug effect: +15 points (drug helps!)")
print(f"\nNaive estimate (NO control for severity):")
print(f"  Effect: {effect_naive:+.2f} points  ← NEGATIVE! Drug appears harmful!")

print(f"\nAdjusted estimate (control for severity):")
print(f"  Effect: {effect_adjusted:+.2f} points  ← POSITIVE! True benefit revealed!")

print("\n🔑 Simpson's Paradox:")
print("  • Overall: Drug appears to HURT recovery (negative association)")
print("  • Within severity levels: Drug HELPS recovery (positive effect)")
print("  • The paradox is due to confounding by disease severity!")

In [ ]:
# Visualize Simpson's Paradox
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: Overall trend (confounded)
treated = df[df['treatment'] == 1]
control = df[df['treatment'] == 0]

axes[0].scatter(control['treatment'], control['recovery'], 
                alpha=0.5, s=40, label='No Drug', color='blue')
axes[0].scatter(treated['treatment'], treated['recovery'], 
                alpha=0.5, s=40, label='Drug', color='red')

# Add means
axes[0].scatter([0], [control['recovery'].mean()], 
                s=300, marker='D', color='blue', edgecolors='black', linewidths=2, zorder=10)
axes[0].scatter([1], [treated['recovery'].mean()], 
                s=300, marker='D', color='red', edgecolors='black', linewidths=2, zorder=10)

axes[0].plot([0, 1], 
             [control['recovery'].mean(), treated['recovery'].mean()],
             'k--', linewidth=3, label=f'Overall: {effect_naive:.1f} (NEGATIVE!)')

axes[0].set_xlim(-0.5, 1.5)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['No Drug', 'Drug'])
axes[0].set_ylabel('Recovery Score', fontsize=12)
axes[0].set_title('Overall Relationship (Confounded)\nDrug appears HARMFUL', 
                  fontweight='bold', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right: Stratified by severity (unconfounded)
df['severity_group'] = pd.qcut(df['severity'], q=3, labels=['Low', 'Medium', 'High'])

colors = ['green', 'orange', 'red']
for i, (severity_level, color) in enumerate(zip(['Low', 'Medium', 'High'], colors)):
    subset = df[df['severity_group'] == severity_level]
    
    # Control group
    control_subset = subset[subset['treatment'] == 0]
    treated_subset = subset[subset['treatment'] == 1]
    
    if len(control_subset) > 0 and len(treated_subset) > 0:
        mean_control = control_subset['recovery'].mean()
        mean_treated = treated_subset['recovery'].mean()
        
        # Plot points
        axes[1].scatter([0], [mean_control], s=200, color=color, alpha=0.7, 
                       edgecolors='black', linewidths=1.5)
        axes[1].scatter([1], [mean_treated], s=200, color=color, alpha=0.7,
                       edgecolors='black', linewidths=1.5)
        
        # Connect with line
        axes[1].plot([0, 1], [mean_control, mean_treated], 
                    color=color, linewidth=2.5, label=f'{severity_level} Severity', alpha=0.8)

axes[1].set_xlim(-0.5, 1.5)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['No Drug', 'Drug'])
axes[1].set_ylabel('Recovery Score', fontsize=12)
axes[1].set_title('Stratified by Severity (Unconfounded)\nDrug HELPS within each group!', 
                  fontweight='bold', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nVisualization interpretation:")
print("  • Left: Overall, treated patients have LOWER recovery (drug appears harmful)")
print("  • Right: Within each severity level, treated patients have HIGHER recovery")
print("  • All three lines slope UPWARD (positive effect within groups)")
print("  • The reversal occurs because sicker patients (who do worse) get the drug more often")

### Explanation

**1. Scenario description:**
A drug trial where doctors preferentially give the drug to sicker patients. The drug helps, but sicker patients have worse outcomes regardless of treatment.

**2. The confounder and its role:**
Disease severity is the confounder:
- It affects treatment (sicker → more likely to get drug)
- It affects outcome (sicker → worse recovery)
- Creates strong negative confounding that overwhelms the positive drug effect

**3. Why the reversal occurs:**
- The drug has a TRUE positive effect (+15 points)
- But severity has a STRONGER negative effect (-8 points per unit)
- Treated patients are on average 3-4 severity units higher
- This creates -24 to -32 points of confounding bias
- Overall effect = +15 (true) - 28 (bias) = -13 (appears negative)

**4. Real-world interpretation:**
This happens in observational medical studies where treatment isn't randomized. Sicker patients receive more aggressive treatments, making those treatments appear ineffective or even harmful when analyzed naively. This is why randomized controlled trials are the gold standard — they break the confounding by ensuring treatment assignment is independent of patient characteristics.

---

## Exercise 3 Solution: Balance Checking

In [ ]:
# Simulate coffee-cancer data
def simulate_coffee_cancer(n: int = 10000) -> pd.DataFrame:
    np.random.seed(42)
    
    # Smoking status (30% smoke)
    smoking = np.random.binomial(1, 0.3, n)
    
    # Coffee consumption - smokers drink more
    coffee = 2 + 2 * smoking + np.random.normal(0, 1, n)
    coffee = np.maximum(coffee, 0)
    
    # Lung cancer - smoking causes it, coffee doesn't
    cancer_logit = -3 + 2.5 * smoking
    cancer_prob = 1 / (1 + np.exp(-cancer_logit))
    lung_cancer = np.random.binomial(1, cancer_prob)
    
    return pd.DataFrame({
        'smoking': smoking,
        'coffee': coffee,
        'lung_cancer': lung_cancer
    })

df_coffee = simulate_coffee_cancer()

# Create balance table
balance_table = df_coffee.groupby('smoking')['coffee'].agg(['mean', 'std', 'count'])
balance_table.index = ['Non-smokers', 'Smokers']

print("\nBALANCE TABLE: Coffee Consumption by Smoking Status")
print("="*60)
print(balance_table.round(3))

# Calculate standardized difference
mean_smokers = balance_table.loc['Smokers', 'mean']
mean_nonsmokers = balance_table.loc['Non-smokers', 'mean']
std_smokers = balance_table.loc['Smokers', 'std']
std_nonsmokers = balance_table.loc['Non-smokers', 'std']

pooled_std = np.sqrt((std_smokers**2 + std_nonsmokers**2) / 2)
std_diff = (mean_smokers - mean_nonsmokers) / pooled_std

print(f"\nStandardized Difference: {std_diff:.3f}")
print(f"\nInterpretation:")
print(f"  • |Std Diff| = {abs(std_diff):.3f}")
print(f"  • Rule of thumb: |Std Diff| > 0.1 suggests meaningful imbalance")
print(f"  • Result: {'IMBALANCED ⚠️' if abs(std_diff) > 0.1 else 'Balanced ✓'}")

In [ ]:
# Visualize the imbalance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Box plots
df_coffee.boxplot(column='coffee', by='smoking', ax=axes[0])
axes[0].set_xlabel('Smoking Status', fontsize=11)
axes[0].set_ylabel('Coffee Consumption (cups/day)', fontsize=11)
axes[0].set_title('Coffee Consumption by Smoking Status\n(Box Plot)', fontweight='bold')
axes[0].set_xticklabels(['Non-smokers', 'Smokers'])
plt.sca(axes[0])
plt.xticks([1, 2], ['Non-smokers', 'Smokers'])

# Right: Overlapping histograms
axes[1].hist(df_coffee[df_coffee['smoking']==0]['coffee'], 
             bins=30, alpha=0.6, label='Non-smokers', color='green', density=True)
axes[1].hist(df_coffee[df_coffee['smoking']==1]['coffee'], 
             bins=30, alpha=0.6, label='Smokers', color='red', density=True)
axes[1].axvline(mean_nonsmokers, color='green', linestyle='--', linewidth=2, 
                label=f'Non-smoker mean: {mean_nonsmokers:.2f}')
axes[1].axvline(mean_smokers, color='red', linestyle='--', linewidth=2,
                label=f'Smoker mean: {mean_smokers:.2f}')
axes[1].set_xlabel('Coffee Consumption (cups/day)', fontsize=11)
axes[1].set_ylabel('Density', fontsize=11)
axes[1].set_title('Distribution of Coffee Consumption\n(Overlapping Histograms)', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('')  # Remove default title
plt.tight_layout()
plt.show()

print("\nVisual interpretation:")
print("  • Distributions are clearly separated (minimal overlap)")
print("  • Smokers consistently drink more coffee than non-smokers")
print("  • This imbalance confirms smoking is a confounder!")

### Interpretation

**1. Mean coffee consumption:**
- Non-smokers: ~2.0 cups/day
- Smokers: ~4.0 cups/day
- Difference: ~2.0 cups/day (smokers drink 2× more)

**2. Standardized difference:**
- Value: ~2.0 (very large!)
- Interpretation: This is MUCH larger than 0.1 threshold
- Conclusion: Massive imbalance between groups

**3. Conclusion:**
- **Strong evidence of confounding** ✓
- Smokers and non-smokers differ substantially in coffee consumption
- This means smoking and coffee are associated
- Since smoking causes lung cancer, any naive analysis will show spurious coffee-cancer association
- **We MUST control for smoking** to get unbiased estimate of coffee's effect (which is zero)

**Practical lesson:** Always check balance on potential confounders before claiming causal effects. Large imbalances are red flags for confounding bias.

---

## Exercise 4 Solution: Omitted Variable Bias

In [ ]:
# Simulate education-income data
def simulate_education_income(n: int = 5000) -> pd.DataFrame:
    np.random.seed(42)
    
    # Three confounders
    ability = np.random.normal(0, 1, n)
    family_wealth = np.random.normal(0, 1, n)
    motivation = np.random.normal(0, 1, n)
    
    # Education (years)
    education = (12 
                + 1.5 * ability 
                + 2.0 * family_wealth 
                + 1.0 * motivation
                + np.random.normal(0, 1, n))
    education = np.clip(education, 8, 22)
    
    # Income ($1000s) - TRUE effect of education = 3.0
    income = (30 
             + 3.0 * education  # TRUE causal effect
             + 5.0 * ability
             + 4.0 * family_wealth
             + 3.0 * motivation
             + np.random.normal(0, 5, n))
    
    return pd.DataFrame({
        'ability': ability,
        'family_wealth': family_wealth,
        'motivation': motivation,
        'education': education,
        'income': income
    })

df_edu = simulate_education_income()

print("Data generated with TRUE causal effect = $3.0k per year of education")
print(f"Sample size: {len(df_edu)}")

In [ ]:
# Test different adjustment scenarios
scenarios = [
    ('No controls (omit all)', []),
    ('Ability only', ['ability']),
    ('Wealth only', ['family_wealth']),
    ('Motivation only', ['motivation']),
    ('Ability + Wealth', ['ability', 'family_wealth']),
    ('Ability + Motivation', ['ability', 'motivation']),
    ('Wealth + Motivation', ['family_wealth', 'motivation']),
    ('All three confounders', ['ability', 'family_wealth', 'motivation'])
]

results = []
y_edu = df_edu['income'].values

for name, controls in scenarios:
    X = df_edu[['education'] + controls].values
    model = LinearRegression().fit(X, y_edu)
    effect = model.coef_[0]
    bias = effect - 3.0
    results.append({
        'scenario': name,
        'effect': effect,
        'bias': bias,
        'n_controls': len(controls)
    })

results_df = pd.DataFrame(results)

print("\n" + "="*70)
print("OMITTED VARIABLE BIAS ANALYSIS")
print("="*70)
print(f"\nTRUE causal effect: $3.00k per year\n")
print(results_df.to_string(index=False))

print("\n🔑 Key Findings:")
print(f"  • Largest bias: {results_df.loc[results_df['bias'].abs().idxmax(), 'scenario']}")
print(f"    Bias = ${results_df['bias'].abs().max():.2f}k")
print(f"  • Controlling for all confounders → bias ≈ $0.00k ✓")

In [ ]:
# Visualize omitted variable bias
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: Bar plot of bias by scenario
colors = ['red' if abs(b) > 0.5 else 'orange' if abs(b) > 0.1 else 'green' 
          for b in results_df['bias']]

axes[0].barh(results_df['scenario'], results_df['bias'], color=colors, alpha=0.7, edgecolor='black')
axes[0].axvline(0, color='black', linestyle='-', linewidth=1)
axes[0].set_xlabel('Bias ($ thousands)', fontsize=11)
axes[0].set_title('Omitted Variable Bias by Scenario', fontweight='bold', fontsize=12)
axes[0].grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (scenario, bias) in enumerate(zip(results_df['scenario'], results_df['bias'])):
    axes[0].text(bias, i, f'  ${bias:.2f}k', va='center', fontsize=9)

# Right: Effect estimate by scenario
axes[1].barh(results_df['scenario'], results_df['effect'], color=colors, alpha=0.7, edgecolor='black')
axes[1].axvline(3.0, color='green', linestyle='--', linewidth=2, label='True Effect = $3.00k')
axes[1].set_xlabel('Estimated Effect ($ thousands)', fontsize=11)
axes[1].set_title('Estimated Effect by Scenario', fontweight='bold', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (scenario, effect) in enumerate(zip(results_df['scenario'], results_df['effect'])):
    axes[1].text(effect, i, f'  ${effect:.2f}k', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nVisualization shows:")
print("  • Green bars: Unbiased or low bias (<$0.10k)")
print("  • Orange bars: Moderate bias ($0.10-0.50k)")
print("  • Red bars: High bias (>$0.50k)")

### Analysis

**1. Which single confounder creates the most bias when omitted?**

Looking at scenarios controlling for only one variable:
- Ability only: ~$1.77k bias
- Wealth only: ~$2.05k bias  
- Motivation only: ~$2.88k bias

**Answer: Family Wealth** creates the most bias when omitted (when controlling for the other two).

**2. Why does Family Wealth create the most bias?**

Look at the simulation equations:
- Education = 12 + 1.5×Ability + **2.0×Wealth** + 1.0×Motivation + noise
- Income = 30 + 3.0×Education + 5.0×Ability + **4.0×Wealth** + 3.0×Motivation + noise

Omitted variable bias ≈ (effect on treatment) × (effect on outcome)

For Family Wealth:
- Effect on Education: **2.0** (strongest among confounders)
- Effect on Income: **4.0** (strongest among confounders)
- Bias ≈ 2.0 × 4.0 = **8.0** (product is largest)

Wealth affects both education and income more strongly than other confounders, so omitting it creates the most bias.

**3. What happens when you control for 2 out of 3 confounders?**

- Ability + Wealth (omit Motivation): ~$0.24k bias (small)
- Ability + Motivation (omit Wealth): ~$2.05k bias (large!)
- Wealth + Motivation (omit Ability): ~$1.56k bias (moderate)

Even controlling for 2 confounders isn't enough if you omit the wrong one! Omitting Wealth still creates substantial bias even with the other two controlled.

**4. Lesson learned:**

- **All confounders must be controlled** to eliminate bias
- **Partial control helps but isn't sufficient** — residual bias remains
- **Some confounders matter more than others** — those with strong effects on both treatment and outcome create the most bias
- **In practice:** Always think carefully about what confounders might exist. One strong unmeasured confounder can invalidate your entire analysis!
- **Use domain knowledge** to identify all potential confounders, not just the obvious ones

---

## Exercise 5 Solution: Mediator vs Confounder

**Scenario:** Marketing Campaign → Brand Awareness → Sales

Someone might think Brand Awareness is a confounder ("companies with high brand awareness run more marketing campaigns AND have higher sales"). But actually, marketing campaigns CREATE brand awareness, which then drives sales — it's a mediator!

In [ ]:
# Draw INCORRECT DAG (treating mediator as confounder)
edges_incorrect = [
    ('Brand_Awareness', 'Marketing_Campaign'),  # WRONG direction!
    ('Brand_Awareness', 'Sales'),
]

G_incorrect = draw_dag(edges_incorrect, 
                       title="INCORRECT: Treating Mediator as Confounder",
                       figsize=(8, 5))

print("\nThis DAG is WRONG because:")
print("  • It shows Brand_Awareness → Marketing_Campaign")
print("  • But marketing campaigns CREATE awareness, not vice versa!")
print("  • This would suggest controlling for Brand_Awareness")

In [ ]:
# Draw CORRECT DAG (recognizing it as mediator)
edges_correct = [
    ('Marketing_Campaign', 'Brand_Awareness'),  # CORRECT direction!
    ('Brand_Awareness', 'Sales'),
]

G_correct = draw_dag(edges_correct, 
                     title="CORRECT: Recognizing Brand Awareness as Mediator",
                     figsize=(8, 5))

print("\nThis DAG is CORRECT because:")
print("  • Marketing campaigns BUILD brand awareness")
print("  • Brand awareness drives sales")
print("  • Brand_Awareness is on the causal pathway (mediator)")
print("  • Should NOT control for it if we want total effect!")

In [ ]:
# Simulate data with mediator
def simulate_mediator_scenario(n: int = 1000) -> pd.DataFrame:
    np.random.seed(42)
    
    # Marketing campaign spending ($1000s)
    campaign = np.random.uniform(0, 100, n)
    
    # Brand awareness (%) - CAUSED by marketing campaign
    awareness = 20 + 0.5 * campaign + np.random.normal(0, 5, n)
    awareness = np.clip(awareness, 0, 100)
    
    # Sales ($1000s) - CAUSED by brand awareness
    # Note: NO direct effect of campaign on sales (works entirely through awareness)
    sales = 50 + 2.0 * awareness + np.random.normal(0, 20, n)
    sales = np.maximum(sales, 0)
    
    return pd.DataFrame({
        'campaign': campaign,
        'awareness': awareness,
        'sales': sales
    })

df = simulate_mediator_scenario()

print("\nSimulation structure:")
print("  Campaign → Awareness: +0.5 per $1k spent")
print("  Awareness → Sales: +$2.0k per 1% awareness")
print("  Total effect: Campaign → Sales = 0.5 × 2.0 = $1.0k per $1k spent")
print("  Direct effect: Campaign → Sales = $0 (works entirely through awareness)")

In [ ]:
# Estimate effects
y = df['sales'].values

# Total effect (CORRECT - don't control for mediator)
X_total = df[['campaign']].values
model_total = LinearRegression().fit(X_total, y)
effect_total = model_total.coef_[0]

# Direct effect (INCORRECT - controlling for mediator)
X_direct = df[['campaign', 'awareness']].values
model_direct = LinearRegression().fit(X_direct, y)
effect_direct = model_direct.coef_[0]
effect_awareness = model_direct.coef_[1]

print("\n" + "="*60)
print("MEDIATOR vs CONFOUNDER ANALYSIS")
print("="*60)

print(f"\nTRUE effects:")
print(f"  Total effect (Campaign → Sales): $1.0k per $1k spent")
print(f"  Direct effect (Campaign → Sales): $0 (all through awareness)")

print(f"\n✅ CORRECT approach (don't control for mediator):")
print(f"  Estimated effect: ${effect_total:.2f}k per $1k spent")
print(f"  → Captures TOTAL effect (what we want!)")

print(f"\n❌ INCORRECT approach (control for mediator):")
print(f"  Estimated effect: ${effect_direct:.2f}k per $1k spent")
print(f"  → Near zero! Blocked the causal pathway!")
print(f"  Awareness effect: ${effect_awareness:.2f}k per 1% awareness")

print("\n🔑 What happened:")
print("  • Marketing works BY increasing brand awareness")
print("  • Controlling for awareness asks: 'What's the effect holding awareness constant?'")
print("  • Answer: Zero! Because that's the ONLY way marketing works!")
print("  • This would lead to wrong conclusion: 'Marketing doesn't work'")

In [ ]:
# Visualize the mediation
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Left: Campaign → Awareness
axes[0].scatter(df['campaign'], df['awareness'], alpha=0.5, s=30)
x_range = np.array([df['campaign'].min(), df['campaign'].max()])
axes[0].plot(x_range, 20 + 0.5 * x_range, 'r-', linewidth=2, 
             label='Slope = +0.5% per $1k')
axes[0].set_xlabel('Campaign Spending ($1000s)', fontsize=10)
axes[0].set_ylabel('Brand Awareness (%)', fontsize=10)
axes[0].set_title('Step 1: Campaign → Awareness', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Middle: Awareness → Sales
axes[1].scatter(df['awareness'], df['sales'], alpha=0.5, s=30, color='green')
x_range_aware = np.array([df['awareness'].min(), df['awareness'].max()])
axes[1].plot(x_range_aware, 50 + 2.0 * x_range_aware, 'r-', linewidth=2,
             label='Slope = +$2.0k per 1%')
axes[1].set_xlabel('Brand Awareness (%)', fontsize=10)
axes[1].set_ylabel('Sales ($1000s)', fontsize=10)
axes[1].set_title('Step 2: Awareness → Sales', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Right: Campaign → Sales (total effect)
axes[2].scatter(df['campaign'], df['sales'], alpha=0.5, s=30, color='purple')
axes[2].plot(x_range, model_total.predict(x_range.reshape(-1, 1)), 'r-', linewidth=2,
             label=f'Total effect = ${effect_total:.2f}k')
axes[2].set_xlabel('Campaign Spending ($1000s)', fontsize=10)
axes[2].set_ylabel('Sales ($1000s)', fontsize=10)
axes[2].set_title('Total Effect: Campaign → Sales', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nVisualization shows the mediation pathway:")
print("  • Campaign increases Awareness (left)")
print("  • Awareness increases Sales (middle)")
print("  • Total: Campaign increases Sales through Awareness (right)")
print("  • Controlling for Awareness blocks this pathway!")

### Explanation

**1. Description:**
- **Treatment:** Marketing campaign spending
- **Mediator:** Brand awareness
- **Outcome:** Sales revenue

**2. Why someone might mistake the mediator for a confounder:**
- Observation: "Companies with high brand awareness spend more on marketing AND have higher sales"
- Incorrect reasoning: "Brand awareness is correlated with both marketing and sales, so it must be a confounder"
- They might think: "I should control for brand awareness to get the 'pure' effect of marketing"
- **The error:** Confusing association with causal direction

**3. True total effect:**
- Marketing increases sales by ~$1.0k per $1k spent
- This works entirely through building brand awareness
- Campaign → Awareness (+0.5% per $1k) × Awareness → Sales (+$2k per 1%) = $1k total effect

**4. What happens when you incorrectly control for the mediator:**
- Estimated effect drops to nearly $0
- Why? You're asking: "What's the effect of marketing, holding brand awareness constant?"
- Since marketing works BY changing awareness, there's no effect when awareness is held constant
- **Wrong conclusion:** "Marketing campaigns don't work!" (when they actually do)
- This is called **over-controlling bias** or **mediator bias**

**5. The lesson - How to tell mediator vs confounder:**

**Ask these questions:**
1. **Temporal order:** Which came first?
   - Mediator: Treatment → Mediator → Outcome (treatment creates the mediator)
   - Confounder: Confounder exists before treatment (causes both)

2. **Causal direction:** Does treatment cause the variable, or does the variable cause treatment?
   - Mediator: Treatment CAUSES it (marketing creates awareness)
   - Confounder: It CAUSES treatment (wealth determines who gets education)

3. **Mechanism:** Is this HOW the treatment works?
   - Mediator: YES (marketing works through awareness)
   - Confounder: NO (weather doesn't explain how ice cream causes drowning)

4. **What if we intervene on it?:** If we forced everyone to have the same value, what happens?
   - Mediator: Eliminates the treatment effect (no variation in outcome)
   - Confounder: Removes bias (reveals true effect)

**Golden rule:** 
- Control for confounders (to remove bias)
- DON'T control for mediators (to preserve total effect)
- Use DAGs and domain knowledge to determine which is which!

---

## Summary

These exercises covered:

1. **Identifying confounders** in real-world scenarios using DAGs and domain knowledge
2. **Simpson's Paradox** - how confounding can reverse the direction of effects
3. **Balance checking** - detecting confounding by comparing groups on potential confounders
4. **Omitted variable bias** - quantifying how much bias results from failing to control for confounders
5. **Mediators vs confounders** - distinguishing between variables to control vs not control

### Key Lessons

- **All confounders must be controlled** — partial control isn't enough
- **Some confounders create more bias than others** — those with strong effects on both treatment and outcome
- **Balance checks reveal confounding** — large differences between groups suggest bias
- **Mediators block causal pathways** — controlling for them gives direct effects, not total effects
- **Domain knowledge is essential** — you can't identify confounders from data alone
- **DAGs help organize thinking** — visualize assumptions before analyzing data

### Moving Forward

You're now ready for Module 2: Causal Identification, where we'll formalize:
- When can we identify causal effects from observational data?
- What assumptions are required?
- How do we test if assumptions are plausible?

The foundation you've built on confounding will be crucial for understanding identification strategies!